# Lecture 2.3: Embeddings and pgvector on Lakebase

Chunks are read from the Delta `plant_care_chunks` table. Embeddings, plant metadata, and PostgreSQL full-text state are stored in Lakebase.

```text
plant_care_chunks (Delta)
      │  ai_query(..., ARRAY<FLOAT>)
      ▼
Lakebase Postgres + pgvector
      ├── HNSW cosine index
      ├── typed and JSONB metadata
      └── PostgreSQL full-text ranking
```

In [1]:
from loguru import logger
from plant_care.config import get_env, load_config
from plant_care.vector_search import PlantCarePgVectorManager
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()
env = get_env()
cfg = load_config("../project_config.yml", env)

logger.info(f"Environment: {env}")
logger.info(f"Source table: {cfg.catalog}.{cfg.schema}.plant_care_chunks")
logger.info(f"Lakebase instance: {cfg.lakebase_instance}")
logger.info(f"Embedding endpoint: {cfg.embedding_endpoint}")

2026-11-18 12:37:34.753 | INFO     | __main__:<module>:5 - Environment: dev
2026-11-18 12:37:34.754 | INFO     | __main__:<module>:6 - Source table: llmops_dev.srinidhi_ilango98.plant_care_chunks
2026-11-18 12:37:34.754 | INFO     | __main__:<module>:7 - Lakebase instance: lb-llmops-srinidhi-ilango98
2026-11-18 12:37:34.755 | INFO     | __main__:<module>:8 - Embedding endpoint: databricks-gte-large-en


In [3]:
# The manager obtains a short-lived Lakebase credential through the Databricks SDK.
pgvector = PlantCarePgVectorManager(config=cfg, spark=spark)
pgvector.ensure_schema()

logger.info(f"Lakebase table: {pgvector.table_name}")

2026-11-18 12:37:40.117 | INFO     | plant_care.vector_search:ensure_schema:142 - Lakebase pgvector table ready: plant_care_embeddings
2026-11-18 12:37:40.118 | INFO     | __main__:<module>:5 - Lakebase table: plant_care_embeddings


## 1. Create the pgvector schema

`ensure_schema()` enables the `vector` extension and creates the embedding table with:

- an HNSW index using cosine distance (`<=>`)
- a GIN index over PostgreSQL `tsvector` data for keyword search
- a `(slug)` index for plant lookups

## 2. Generate and load embeddings

Chunks are embedded only when the table is empty. If rows already exist, `index_delta_chunks()` skips embedding and returns 0. Set `refresh = True` to re-embed every chunk. Re-runs are safe because the upsert is keyed on the chunk `id`, and plant metadata is refreshed on that run.

In [4]:
refresh = False  # Set to True to re-embed every chunk
indexed_count = pgvector.index_delta_chunks(refresh=refresh)
logger.info(f"Chunks indexed: {indexed_count}")

2026-11-18 12:37:47.888 | INFO     | plant_care.vector_search:ensure_schema:142 - Lakebase pgvector table ready: plant_care_embeddings
2026-11-18 12:37:50.326 | INFO     | plant_care.vector_search:index_delta_chunks:236 - plant_care_embeddings already populated; skipping embedding (pass refresh=True to rebuild)
2026-11-18 12:37:50.326 | INFO     | __main__:<module>:3 - Chunks indexed: 0


## 3. Semantic search

The query is embedded with the same endpoint as the documents. PostgreSQL ranks rows by cosine distance and returns the highest-scoring chunks.

In [5]:
query = 'Which houseplants are safe around cats and grow well in low light?'
results = pgvector.search(query=query, num_results=5)

logger.info(f"Query: {query}")
for position, row in enumerate(results, 1):
    logger.info(f"{position}. {row.get('slug', 'N/A')} | {row.get('section', 'N/A')} | score={float(row['score']):.4f}")
    logger.info(f"   {row.get('text', '')[:250]}...")

2026-11-18 12:39:32.756 | INFO     | __main__:<module>:4 - Query: Which houseplants are safe around cats and grow well in low light?
2026-11-18 12:39:32.756 | INFO     | __main__:<module>:6 - 1. cast-iron-plant | Light | score=0.6921
2026-11-18 12:39:32.762 | INFO     | __main__:<module>:7 -    Light: This is the ultimate low-light houseplant. It is the perfect candidate for that dark corner of the bedroom or the windowless office cubicle that relies on fluorescent lighting. While it will survive in deep shade, it will grow slightly faster ...
2026-11-18 12:39:32.762 | INFO     | __main__:<module>:6 - 2. dracaena-marginata | Light: Preventing Stretching and Leaf Loss | score=0.6811
2026-11-18 12:39:32.762 | INFO     | __main__:<module>:7 -    Light: Preventing Stretching and Leaf Loss: Dracaena marginata is often sold as a low-light plant. It will survive in dim rooms, but it grows slowly there, stretches toward the light, and sheds lower leaves. Bright, filtered, indirect light gives 

## 4. Semantic search with metadata filters

Filters are limited to a whitelist: `slug`, `tags`, `category`, `difficulty`, `family`, `pet_safe_cat`, and `pet_safe_dog`. Unknown keys raise an error.

In [6]:
query = 'low light plant for an office'
filtered_results = pgvector.search(query=query, num_results=3, filters={'pet_safe_cat': True})

logger.info(f"Query: {query} | pet_safe_cat=True")
for position, row in enumerate(filtered_results, 1):
    logger.info(f"{position}. {row.get('slug', 'N/A')} | pet_safe_cat={row.get('pet_safe_cat')} | score={float(row['score']):.4f}")

2026-11-18 12:39:40.711 | INFO     | __main__:<module>:4 - Query: low light plant for an office | pet_safe_cat=True
2026-11-18 12:39:40.721 | INFO     | __main__:<module>:6 - 1. cast-iron-plant | pet_safe_cat=True | score=0.7475
2026-11-18 12:39:40.721 | INFO     | __main__:<module>:6 - 2. java-fern | pet_safe_cat=True | score=0.6930
2026-11-18 12:39:40.724 | INFO     | __main__:<module>:6 - 3. nerve-plant | pet_safe_cat=True | score=0.6632


## 5. Hybrid search: vector similarity + full-text ranking

Hybrid search combines semantic similarity with keyword coverage. The keyword score is the fraction of distinct query words that appear in the chunk, so repeated words do not raise the score. This helps with exact terms, such as plant names, that semantic search alone can underweight.

In [15]:
query = 'How to take care of Pothos?'
hybrid_results = pgvector.hybrid_search(query=query, num_results=5)

logger.info(f"Hybrid query: {query}")
for position, row in enumerate(hybrid_results, 1):
    logger.info(
        f"{position}. {row.get('slug', 'N/A')} | score={float(row['score']):.4f} | "
        f"vector={float(row['vector_score']):.4f} | text={float(row['text_score']):.4f}"
    )

2026-11-18 12:45:41.457 | INFO     | __main__:<module>:4 - Hybrid query: How to take care of Pothos?
2026-11-18 12:45:41.458 | INFO     | __main__:<module>:6 - 1. sarracenia | score=0.5359 | vector=0.6942 | text=0.1667
2026-11-18 12:45:41.459 | INFO     | __main__:<module>:6 - 2. golden-pothos-care | score=0.5247 | vector=0.7106 | text=0.0909
2026-11-18 12:45:41.460 | INFO     | __main__:<module>:6 - 3. golden-pothos-care | score=0.5217 | vector=0.7063 | text=0.0909
2026-11-18 12:45:41.460 | INFO     | __main__:<module>:6 - 4. peperomia-obtusifolia | score=0.5067 | vector=0.6849 | text=0.0909
2026-11-18 12:45:41.461 | INFO     | __main__:<module>:6 - 5. epipremnum-marble-queen | score=0.4934 | vector=0.6059 | text=0.2308


## 6. Inspect the Lakebase index

In [8]:
with pgvector.connection() as conn:
    table_count = conn.execute(f"SELECT count(*) FROM {pgvector.table_name}").fetchone()[0]
    indexes = conn.execute(
        """
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = %s
        ORDER BY indexname
        """,
        (pgvector.table_name,),
    ).fetchall()

logger.info(f"Indexed rows: {table_count}")
for index_name, definition in indexes:
    logger.info(f"{index_name}: {definition}")

2026-11-18 12:40:03.932 | INFO     | __main__:<module>:13 - Indexed rows: 1644
2026-11-18 12:40:03.933 | INFO     | __main__:<module>:15 - plant_care_embeddings_embedding_hnsw: CREATE INDEX plant_care_embeddings_embedding_hnsw ON public.plant_care_embeddings USING hnsw (embedding vector_cosine_ops)
2026-11-18 12:40:03.933 | INFO     | __main__:<module>:15 - plant_care_embeddings_pkey: CREATE UNIQUE INDEX plant_care_embeddings_pkey ON public.plant_care_embeddings USING btree (id)
2026-11-18 12:40:03.934 | INFO     | __main__:<module>:15 - plant_care_embeddings_search_gin: CREATE INDEX plant_care_embeddings_search_gin ON public.plant_care_embeddings USING gin (search_vector)
2026-11-18 12:40:03.934 | INFO     | __main__:<module>:15 - plant_care_embeddings_slug: CREATE INDEX plant_care_embeddings_slug ON public.plant_care_embeddings USING btree (slug)


## Summary

1. Read chunks from the Delta `plant_care_chunks` table.
2. Generate 1024-dimensional embeddings with Databricks `ai_query`.
3. Embed and upsert chunks, metadata, and vectors into Lakebase Postgres. This runs when the table is empty, or on every run with `refresh = True`.
4. Use an HNSW cosine index for semantic retrieval.
5. Combine vector similarity, whitelisted metadata filters, and full-text ranking in SQL.